In [1]:
import pickle
import pandas as pd

In [8]:
model_path  = '../2_model_development/model_development_results_dict.pkl'
data_prefix = '../3_model_evaluation/'
non_feature_cols = ['SMILES', 'MP', 'Type', 'Ro5']
model_types = ['RF', 'XGB', 'LGB']

In [3]:
with open(model_path, 'rb') as f:
    model_development_results_dict = pickle.load(f)

In [26]:
import numpy as np
import lightgbm as lgb
import xgboost as xgb
from sklearn.ensemble import RandomForestRegressor

_NON_FEATURE_COLS = ['SMILES', 'MP', 'Type', 'Ro5', 'MP_pred',
                     'pred_lower', 'pred_upper', 'uncertainty']


def _get_feature_cols(model, data):
    """
    Return the feature column names to use for prediction.
    Prefers model-stored names; falls back to dropping known non-feature columns.
    """
    model_class = type(model).__name__
    try:
        if model_class == 'RandomForestRegressor':
            return list(model.feature_names_in_)
        elif model_class == 'LGBMRegressor':
            return model.booster_.feature_name()
        elif model_class == 'XGBRegressor':
            return list(model.get_booster().feature_names)
    except AttributeError:
        pass
    # Fallback: drop any known non-feature columns present in data
    return [c for c in data.columns if c not in _NON_FEATURE_COLS]


def model_confidence_analysis(model, test_data, train_data=None):
    """
    Estimate prediction uncertainty and attach it to test_data.

    RF  → σ = std of individual tree predictions (train_data not needed)
    LGB → fits quantile regressors (alpha=0.1 / 0.9) on train_data
    XGB → fits quantile regressors (alpha=0.1 / 0.9) on train_data

    Parameters
    ----------
    model      : fitted RF, LGBMRegressor, or XGBRegressor
    test_data  : pd.DataFrame with non_feature_cols = ['SMILES', 'MP', 'Type', 'Ro5']
    train_data : pd.DataFrame (required for LGB / XGB)

    Returns
    -------
    result : pd.DataFrame – copy of test_data with columns added:
        RF      → 'uncertainty'  (std of tree predictions)
        LGB/XGB → 'pred_lower' (Q10), 'pred_upper' (Q90), 'uncertainty' (Q90 - Q10)
    """
    feature_cols = _get_feature_cols(model, test_data)
    X_test = test_data[feature_cols]
    result = test_data.copy()
    model_class = type(model).__name__

    # ── Random Forest: std across individual tree predictions ──────────
    if model_class == 'RandomForestRegressor':
        tree_preds = np.array([tree.predict(X_test) for tree in model.estimators_])
        result['uncertainty'] = tree_preds.std(axis=0)

    # ── LightGBM: quantile regression at alpha=0.1 and 0.9 ────────────
    elif model_class == 'LGBMRegressor':
        if train_data is None:
            raise ValueError("train_data must be provided for LGB quantile uncertainty.")
        X_train = train_data[feature_cols]
        y_train = train_data['MP'].values

        params = model.get_params()
        params.pop('objective', None)
        params.pop('verbose', None)

        lower_model = lgb.LGBMRegressor(**params, objective='quantile', alpha=0.1, verbose=-1)
        upper_model = lgb.LGBMRegressor(**params, objective='quantile', alpha=0.9, verbose=-1)
        lower_model.fit(X_train, y_train)
        upper_model.fit(X_train, y_train)

        result['pred_lower']  = lower_model.predict(X_test)
        result['pred_upper']  = upper_model.predict(X_test)
        result['uncertainty'] = result['pred_upper'] - result['pred_lower']

    # ── XGBoost: quantile regression at alpha=0.1 and 0.9 ─────────────
    elif model_class == 'XGBRegressor':
        if train_data is None:
            raise ValueError("train_data must be provided for XGB quantile uncertainty.")
        X_train = train_data[feature_cols]
        y_train = train_data['MP'].values

        params = model.get_params()
        params.pop('objective', None)

        lower_model = xgb.XGBRegressor(**params, objective='reg:quantileerror',
                                        quantile_alpha=0.1, verbosity=0)
        upper_model = xgb.XGBRegressor(**params, objective='reg:quantileerror',
                                        quantile_alpha=0.9, verbosity=0)
        lower_model.fit(X_train, y_train)
        upper_model.fit(X_train, y_train)

        result['pred_lower']  = lower_model.predict(X_test)
        result['pred_upper']  = upper_model.predict(X_test)
        result['uncertainty'] = result['pred_upper'] - result['pred_lower']

    else:
        raise ValueError(
            f"Unsupported model type '{model_class}'. "
            "Expected RandomForestRegressor, LGBMRegressor, or XGBRegressor."
        )

    return result


In [27]:
feature_data_prefix = '../0_data/processed_data/'

confidence_results = {}
for model_name in model_types:
    model = model_development_results_dict[model_name]['best_model_info']['model']

    # evaluation CSV only has test rows → use as test_data
    test_data = pd.read_csv(data_prefix + f"model_evaluation_results_{model_name}.csv")

    # original scaled parquet has both Train/Test → use Train rows for quantile fitting
    original_data = pd.read_parquet(feature_data_prefix + f"data_with_selected_features_{model_name}_scaled.parquet")
    train_data = original_data[original_data['Type'] == 'Train']

    result = model_confidence_analysis(model, test_data, train_data=train_data)
    confidence_results[model_name] = result
    print(f"{model_name}: done — {result.shape[0]} test rows, uncertainty column added.")

confidence_results


RF: done — 5166 test rows, uncertainty column added.


TypeError: type object got multiple values for keyword argument 'verbosity'

In [12]:
test_data

,SMILES,MP,Type,Ro5,RDKit_TPSA,RDKit_FpDensityMorgan3,RDKit_NumRotatableBonds,RDKit_VSA_EState4,RDKit_BertzCT,RDKit_BCUT2D_MRHI,...,RDKit_NumHAcceptors,RDKit_fr_Ar_NH,MACCS_101,RDKit_fr_ArN,RDKit_fr_imidazole,RDKit_Chi3n,RDKit_PEOE_VSA2,RDKit_ExactMolWt,RDKit_EState_VSA9,MP_pred
0,[O-][N+](=O)c1c(C)c(C(=O)C)c(c(c1C(C)(C)C)[N+]...,135.5,Test,1,1.469587,-1.430020,0.035715,-0.979718,0.442559,-0.352389,...,0.921572,-0.199183,-0.618592,-0.268728,-0.13318,0.296533,3.254063,0.406609,-0.817685,144.792644
1,CN(NC(=O)CCC(=O)O)C,154.5,Test,1,0.507304,0.089508,0.337194,0.224807,-1.085124,-0.486108,...,-0.005615,-0.199183,-0.618592,-0.268728,-0.13318,-0.922830,1.636929,-0.857210,-0.327823,95.467196
2,CCCCc1ccc(cc1)NC(=O)Oc1ccc(cc1)OC,143.0,Test,1,-0.122990,0.002796,0.940151,0.072728,0.360855,-0.444646,...,-0.005615,-0.199183,-0.618592,-0.268728,-0.13318,0.241667,0.069922,0.454041,0.091116,122.084054
3,OC(=O)COCCN1C(=O)c2c(C1=O)cccc2,128.0,Test,1,0.914654,-0.006839,0.638672,-0.362995,-0.020869,-0.208970,...,0.457979,-0.199183,1.616573,-0.268728,-0.13318,-0.114470,1.551958,-0.018237,0.126577,197.996066
4,CCCCCCCCCCCCCCC,10.0,Test,1,-1.480634,-2.847146,2.749022,-0.626227,-1.323539,-1.374480,...,-1.396396,-0.199183,-0.618592,-0.268728,-0.13318,0.201356,-0.789161,-0.365346,-0.817685,32.083812
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5161,c1ccc2c(c1)c1cc3ccc4c(c3nc1cc2)cccc4,226.0,Test,1,-1.112677,-0.430765,-0.868720,0.124226,1.976777,-0.259144,...,-0.932802,-0.199183,1.616573,-0.268728,-0.13318,0.791899,-0.789161,0.265017,-0.339579,143.350090
5162,COc1cc(OC)cc(c1C#N)OC,142.0,Test,1,-0.011090,-0.839552,0.035715,-0.496212,-0.462579,-0.642258,...,0.457979,-0.199183,-0.618592,-0.268728,-0.13318,-0.475385,-0.789161,-0.546159,1.050281,93.594717
5163,OCCCCCCCCCCc1ccccc1,36.0,Test,1,-0.903150,-0.879629,2.146065,-0.111926,-0.753977,-0.927824,...,-0.932802,-0.199183,-0.618592,-0.268728,-0.13318,0.205908,-0.789161,-0.158401,-0.327823,63.903770
5164,Clc1c2OC3Cc4c(C3Oc2c(c(c1Cl)Cl)Cl)cccc4,159.0,Test,1,-0.953676,-0.158241,-0.868720,0.180255,0.923148,0.004288,...,-0.469209,-0.199183,1.616573,-0.268728,-0.13318,0.500622,-0.789161,1.027089,4.542560,117.117273
